In [1]:
import sys
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import pandas as pd
from requests import HTTPError

# Add project root to path (if needed for shared utils later)
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

In [3]:
def generate_211_url(latitude, longitude, location, topic_path):
    base_url = "https://211ontario.ca/results/"
    return (
        f"{base_url}?latitude={latitude}&longitude={longitude}&searchLocation={location}"
        f"&searchTerms=&exct=0&sd=25&ss=Distance&topicPath={topic_path}"
    )

In [4]:
import json


ontario_city_coordinates = json.loads(Path("ontario_city_coordinates.json").read_text())

In [5]:
ontario_city_coordinates

{'Akwesasne': {'lat': 45.0047, 'lon': -74.733},
 'Alexandria': {'lat': 45.3179, 'lon': -74.6398},
 'Alliston': {'lat': 44.1539, 'lon': -79.8692},
 'Armstrong': {'lat': 50.3021, 'lon': -89.0374},
 'Atikokan': {'lat': 48.7625, 'lon': -91.6246},
 'Ayr': {'lat': 43.2855, 'lon': -80.4507},
 'Bala': {'lat': 45.0177, 'lon': -79.617},
 'Barrie': {'lat': 44.3893, 'lon': -79.6901},
 'Batchewana': {'lat': 47.1749, 'lon': -84.2883},
 'Beachburg': {'lat': 45.7329, 'lon': -76.8559},
 'Beaverton': {'lat': 44.4296, 'lon': -79.155},
 'Belleville': {'lat': 44.2436, 'lon': -77.3608},
 'Blind River': {'lat': 46.1884, 'lon': -82.9564},
 'Blyth': {'lat': 43.7366, 'lon': -81.4294},
 'Bourget': {'lat': 45.4349, 'lon': -75.1579},
 'Bracebridge': {'lat': 45.0415, 'lon': -79.311},
 'Brampton': {'lat': 43.6858, 'lon': -79.7599},
 'Brantford': {'lat': 43.1408, 'lon': -80.2632},
 'Brockville': {'lat': 44.6039, 'lon': -75.7017},
 'Brooklin': {'lat': 43.9587, 'lon': -78.9601},
 'Calstock': {'lat': 49.7884, 'lon': -84

In [7]:
# Browser automation with Playwright to pass Turnstile and paginate
# If not installed, run in a separate cell/terminal: pip install playwright && playwright install
import asyncio
from urllib.parse import urljoin
from playwright.async_api import async_playwright

def parse_records(html, source_url):
    soup = BeautifulSoup(html, "html.parser")
    resources = []
    for record in soup.select("div.record"):
        title_el = record.select_one(".title a")
        title = title_el.get_text(strip=True) if title_el else None
        subtitle_el = record.select_one(".subtitle")
        subtitle = subtitle_el.get_text(strip=True) if subtitle_el else None
        city_el = record.select_one(".subsubtitle")
        city = city_el.get_text(strip=True) if city_el else None
        phone_numbers = [a.get_text(strip=True) for a in record.select("span.linkphone_fa6 a")]
        website_el = record.select_one("a.linkexternal_fa6")
        website = website_el["href"] if website_el and website_el.has_attr("href") else None
        record_info = record.find_next_sibling("div", class_="record-info")
        address = None
        if record_info:
            info_text = " ".join(record_info.stripped_strings)
            address = info_text.replace("(0km)", "").strip() if info_text else None
        details = record.find_next_sibling("div", class_="record-list-details")
        description = None
        if details:
            desc_el = details.select_one(".record-list-desc")
            description = desc_el.get_text(" ", strip=True) if desc_el else None
        resources.append({
            "name": title,
            "program": subtitle,
            "address": address,
            "city": city,
            "phone": ", ".join(phone_numbers) if phone_numbers else None,
            "website": website,
            "description": description,
            "source_url": source_url
        })
    return resources

async def fetch_all_pages(start_url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()
        await page.goto(start_url, wait_until="networkidle")
        print("If a Turnstile challenge appears, complete it in the browser window.")
        await page.wait_for_timeout(10000)
        html = await page.content()
        soup = BeautifulSoup(html, "html.parser")
        pagination_links = []
        for a in soup.select("div.paging a[href]"):
            href = a.get("href")
            if not href:
                continue
            pagination_links.append(urljoin(start_url, href))
        # Ensure first page included and dedupe while keeping order
        all_pages = [start_url] + pagination_links
        seen = set()
        ordered_pages = []
        for u in all_pages:
            if u not in seen:
                seen.add(u)
                ordered_pages.append(u)
        results = []
        for u in ordered_pages:
            await page.goto(u, wait_until="networkidle")
            await page.wait_for_timeout(3000)
            results.append((u, await page.content()))
        await browser.close()
        return results

# Loop through all Ontario cities for every disability topic path and concatenate all results
topic_paths = [
    69
    
    # Add additional disability topic paths here, e.g. 130, 137, ...
]
pause_seconds = 4
all_records = []

for topic_path in topic_paths:
    print(f"\n=== Processing topic_path={topic_path} ===")
    for location, coords in ontario_city_coordinates.items():
        latitude = coords.get("lat")
        longitude = coords.get("lon")

        if latitude is None or longitude is None:
            print(f"{location}: skipped (missing coordinates)")
            continue

        url = generate_211_url(latitude, longitude, location, topic_path)
        pages = await fetch_all_pages(url)
        print(f"{location}: Pages fetched: {len(pages)}")

        for page_url, html in pages:
            # Save latest page HTML if you want to inspect
            output_path = project_root / "test" / "extraction_test" / "ontario_211_results.html"
            output_path.write_text(html, encoding="utf-8")

            records = parse_records(html, page_url)
            for rec in records:
                rec["topic_path"] = topic_path
                rec["search_city"] = location
            all_records.extend(records)

        # Stop gap to avoid flooding servers
        await asyncio.sleep(pause_seconds)

df = pd.DataFrame(all_records)
csv_path = project_root / "test" / "extraction_test" / "ontario_211_disability_69.csv"
df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)
print("Total rows:", len(df))
print("Total topics scraped:", df["topic_path"].nunique() if not df.empty else 0)


=== Processing topic_path=69 ===
If a Turnstile challenge appears, complete it in the browser window.
Akwesasne: Pages fetched: 1
If a Turnstile challenge appears, complete it in the browser window.
Alexandria: Pages fetched: 1
If a Turnstile challenge appears, complete it in the browser window.
Alliston: Pages fetched: 1
If a Turnstile challenge appears, complete it in the browser window.
Armstrong: Pages fetched: 1
If a Turnstile challenge appears, complete it in the browser window.
Atikokan: Pages fetched: 1
If a Turnstile challenge appears, complete it in the browser window.
Ayr: Pages fetched: 1
If a Turnstile challenge appears, complete it in the browser window.
Bala: Pages fetched: 1
If a Turnstile challenge appears, complete it in the browser window.
Barrie: Pages fetched: 1
If a Turnstile challenge appears, complete it in the browser window.
Batchewana: Pages fetched: 1
If a Turnstile challenge appears, complete it in the browser window.
Beachburg: Pages fetched: 1
If a Turns

In [3]:
import time
import json
from geopy.geocoders import Nominatim

# Your fully cleaned and deduplicated list
cleaned_cities = [
    "Akwesasne", "Alexandria", "Alliston", "Armstrong", "Atikokan", "Ayr", "Bala", "Barrie", 
    "Batchewana", "Beachburg", "Beaverton", "Belleville", "Blind River", "Blyth", "Bourget", 
    "Bracebridge", "Brampton", "Brantford", "Brockville", "Brooklin", "Calstock", "Cambridge", 
    "Campbellford", "Cannington", "Cat Lake", "Chatham", "Chatham-Kent", "Chelmsford", 
    "Cobourg", "Cochrane", "Collingwood", "Cornwall", "Crysler", "Cutler", "Deseronto", 
    "Don Mills", "Dryden", "Earlton", "East York", "Elliot Lake", "Embrun", "Espanola", 
    "Etobicoke", "Forest", "Fort Erie", "Fort Frances", "Garden River", "Georgetown", 
    "Gogama", "Grand Bend", "Grassy Narrows", "Guelph", "Hagersville", "Hamilton", "Hanmer", 
    "Hensall", "Heron Bay", "Hiawatha", "Hornby", "Huntsville", "Ignace", "Kapuskasing", 
    "Kashechewan", "Kenora", "Kettle Point", "Kettle and Stoney Point", "Killaloe", "Kingston", 
    "Kirkland Lake", "Kitchener", "Larder Lake", "Leamington", "Lindsay", "Little Current", 
    "London", "Longlac", "M'Chigeeng", "Marathon", "Markdale", "Markham", "Massey", "Mattawa", 
    "Merrickville", "Midland", "Migisi Sahgaigan", "Minden", "Mississauga", "Mississauga First Nation", 
    "Moose Factory", "Moosonee", "Muncey", "Muskrat Dam", "Napanee", "Naughton", "Nepean", 
    "New Liskeard", "Newmarket", "Neyaashiinigmiing", "Niagara Falls", "Nipigon", "Noelville", 
    "North Bay", "North York", "Oakville", "Ohsweken", "Orillia", "Oshawa", "Ottawa", 
    "Owen Sound", "Parry Sound", "Penetanguishene", "Perth", "Peterborough", "Pickering", 
    "Picton", "Port Colborne", "Port Hope", "Port Perry", "Portland", "Sarnia", "Sault Ste. Marie", 
    "Scarborough", "Sharon", "Sioux Lookout", "Smiths Falls", "Southampton", "Southwold", 
    "St. Catharines", "St. Charles", "St. Jacobs", "St. Thomas", "Stouffville", "Sturgeon Falls", 
    "Sudbury", "Temagami", "Thamesville", "Thedford", "Thessalon", "Thunder Bay", "Tillsonburg", 
    "Timmins", "Toronto", "Trenton", "Tweed", "Tyendinaga Mohawk Territory", "Uxbridge", 
    "Vaughan", "Victoria Harbour", "Virginiatown", "Walkerton", "Wallaceburg", "Warren", 
    "Wasaga Beach", "Welland", "Wellesley", "West Lorne", "West Nipissing", "Westport", 
    "Whitby", "Wikwemikong", "Windsor", "Woodstock"
]

# Initialize the Nominatim API
geolocator = Nominatim(user_agent="ontario_scraper_bot")

ontario_coordinates = {}

print(f"Starting geocoding for {len(cleaned_cities)} locations. This will take ~2 minutes...")

for city in cleaned_cities:
    # Appending context ensures we don't accidentally get Paris, France instead of Paris, ON!
    query = f"{city}, Ontario, Canada"
    try:
        location = geolocator.geocode(query)
        if location:
            ontario_coordinates[city] = {
                "lat": round(location.latitude, 4),
                "lon": round(location.longitude, 4)
            }
        else:
            ontario_coordinates[city] = {"lat": None, "lon": None}
            print(f"⚠️ Could not find coordinates for: {city}")
            
    except Exception as e:
        print(f"Error on {city}: {e}")
        ontario_coordinates[city] = {"lat": None, "lon": None}
        
    # Nominatim requires a 1-second delay between requests to prevent rate-limiting
    time.sleep(1)

# Save the final dictionary structure
with open("ontario_city_coordinates.json", "w") as f:
    json.dump(ontario_coordinates, f, indent=4)

print("Done! Saved to ontario_city_coordinates.json")

Starting geocoding for 162 locations. This will take ~2 minutes...
⚠️ Could not find coordinates for: Kettle and Stoney Point
Done! Saved to ontario_city_coordinates.json
